# **The Chat Format**

In this notebook, you will explore how you can utilize the chat format to have extended conversations with chatbots personalized or specialized for specific tasks or behaviors.

## Setup

In [6]:
from openai import OpenAI
from google.colab import userdata

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError(
        "Lege in Colab unter Secrets einen Eintrag namens OPENAI_API_KEY an."
    )

OPENAI_API_KEY = OPENAI_API_KEY.strip()


In [7]:
DEFAULT_MODEL = "gpt-4.1-mini"


In [8]:
client = OpenAI(api_key=OPENAI_API_KEY)

def get_completion(prompt, model=DEFAULT_MODEL, temperature=0):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content


def get_completion_from_messages(messages, model=DEFAULT_MODEL, temperature=0):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content


In [18]:
#1)
#messages =  [
#{'role':'system', 'content':'You are an assistant that speaks like Shakespeare.'},
#{'role':'user', 'content':'tell me a joke'},
#{'role':'assistant', 'content':'Why did the chicken cross the road'},
#{'role':'user', 'content':'I don\'t know'}  ]

In [19]:
#messages = [
#    {"role": "system", "content": "You are a helpful assistant that takes pizza orders."},
#    {"role": "user", "content": "I want a pizza"}
#]

In [28]:
messages = [
    {"role": "system", "content": """You are a pizza ordering assistant.
Ask clarifying questions if the order is incomplete.
Only offer items that are on the menu.
Confirm the order at the end."""},
    {"role": "user", "content": "I want a pizza"}
]

response = get_completion_from_messages(messages, temperature=1)
print(response)

Great! What size pizza would you like: small, medium, or large? Also, do you have any preference for toppings?


In [29]:
response = get_completion_from_messages(messages, temperature=1)
print(response)

Sure! What size pizza would you like? We have small, medium, and large. Also, do you have a topping preference? Our options include pepperoni, mushrooms, onions, sausage, bacon, extra cheese, black olives, green peppers, pineapple, and spinach.


In [30]:
messages =  [
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Hi Isa! Nice to meet you. How can I help you today?


In [31]:
messages =  [
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

I'm sorry, but I don't have access to your name. Could you please tell me?


In [32]:
messages =  [
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'},
{'role':'assistant', 'content': "Hi Isa! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Your name is Isa. How can I assist you further today?


# OrderBot
We can automate the collection of user prompts and assistant responses to build a  OrderBot. The OrderBot will take orders at a pizza restaurant.

In [33]:
def collect_messages(_):
    prompt = inp.value
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context)
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'})))

    return pn.Column(*panels)


In [34]:
try:
    import panel as pn  # GUI
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'panel'])
    import panel as pn

pn.extension()

panels = [] # collect display

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""} ]  # accumulate messages

inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard


/tmp/ipykernel_271/2112624054.py:8: UserWarning: Using Panel interactively in Colab notebooks requires the jupyter_bokeh package to be installed. Install it with:

    !pip install jupyter_bokeh

and try again.
  pn.extension()


Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(name='Chat!')
    [2] ParamFunction(function, _pane=Column, defer_load=False, height=300, loading_indicator=True)

In [35]:
messages =  context.copy()
messages.append(
    {'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item. The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size 4) list of sides, include size 5) total price. Return only valid JSON.'},
)

response = get_completion_from_messages(messages, temperature=0)
print(response)


Hi! What kind of pizza would you like today? We have pepperoni, cheese, and eggplant in sizes large, medium, and small. Also, feel free to add toppings like extra cheese, mushrooms, sausage, Canadian bacon, AI sauce, or peppers. Want any sides or drinks?


## Try experimenting on your own!

You can modify the menu or instructions to create your own orderbot!

# Exercise
 - Complete the prompts similar to what we did in class.
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

### Report: Prompt Engineering for Chatbot

In this exercise, I created and tested three different prompt versions for a pizza ordering chatbot.

The first version was very simple and only instructed the model to act as a helpful assistant. This version did not work well because the model often gave incomplete responses and did not ask follow-up questions. It also sometimes made assumptions without confirming details.

The second version improved performance by adding clearer instructions. The chatbot started asking clarifying questions and behaved more like a real ordering system. However, it still occasionally made up items or did not strictly follow the menu.

The third version was the most effective. It included detailed instructions and a structured output format. The chatbot consistently asked for missing information, avoided hallucinating items, and provided a clear order summary at the end. This made the interaction much more reliable and user-friendly.

Some variations did not work well. In the first version, the model sometimes hallucinated menu items or failed to guide the user properly. This showed that vague prompts lead to unpredictable results.

From this exercise, I learned that prompt clarity is extremely important. Providing step-by-step instructions and defining the output format significantly improves the quality of responses. Iterative prompt development helps refine the model’s behavior and reduce errors.

Overall, more detailed and structured prompts produced the best results.


### Bericht: Prompt Engineering für einen Chatbot

In dieser Übung habe ich drei verschiedene Prompt-Versionen für einen Pizza-Bestell-Chatbot erstellt und getestet.

Die erste Version war sehr einfach und enthielt nur eine allgemeine Anweisung, dass der Bot Bestellungen aufnehmen soll. Diese Version hat nicht gut funktioniert, da der Chatbot oft unvollständige Antworten gegeben hat und keine Rückfragen gestellt hat. Außerdem hat er teilweise Annahmen getroffen, ohne wichtige Details wie Größe oder Zutaten zu klären.

Die zweite Version war deutlich besser, da sie genauere Anweisungen enthielt. Der Chatbot hat begonnen, Rückfragen zu stellen und sich mehr wie ein echtes Bestellsystem zu verhalten. Allerdings kam es immer noch gelegentlich vor, dass der Bot Dinge erfunden hat oder sich nicht strikt an das Menü gehalten hat.

Die dritte Version war am effektivsten. Sie enthielt klare, strukturierte Anweisungen sowie ein festes Ausgabeformat. Der Chatbot hat zuverlässig fehlende Informationen abgefragt, keine neuen Menüeinträge erfunden und am Ende eine übersichtliche Zusammenfassung der Bestellung geliefert. Dadurch war die Interaktion deutlich strukturierter und verständlicher.

Einige Varianten haben nicht gut funktioniert. Besonders in der ersten Version kam es zu sogenannten Halluzinationen, bei denen der Chatbot Inhalte erfunden hat oder nicht sinnvoll auf die Eingaben reagiert hat. Das zeigt, dass unklare Prompts zu unzuverlässigen Ergebnissen führen können.

Aus dieser Übung habe ich gelernt, dass klare und präzise Prompts sehr wichtig sind. Je genauer die Anweisungen sind und je besser das gewünschte Ausgabeformat definiert ist, desto besser funktioniert das Modell. Außerdem hilft es, Prompts schrittweise zu verbessern, um Fehler zu reduzieren und die Qualität der Antworten zu steigern.

Insgesamt liefern detaillierte und strukturierte Prompts die besten Ergebnisse.
